In [ ]:
from discrete_tools import *
from sympy import symbols, Poly, factorint, binomial, expand, Integer, simplify, factor, factorint, pretty, latex
from math import comb, factorial
import math
import sympy as sp

x, y = symbols('x y')

# Reference & Tools

## Derangements

$D_n$ = number of permutations where **no element** is in its original position.
$$D_n = n! \sum_{k=0}^{n} \frac{(-1)^k}{k!} \approx \frac{n!}{e}$$

In [ ]:
# Count derangements
for n in range(1, 8):
    print(f"D({n}) = {derangements_count(n)}")

# List all derangements of a string
items = 'ABCD'
d = get_derangements(items)
print(f"\nDerangements of '{items}': {len(d)} total")
print(d)

# Check a specific permutation
original = 'ABCDE'
candidate = 'BAEDC'
is_d = all(candidate[i] != original[i] for i in range(len(original)))
print(f"\nIs '{candidate}' a derangement of '{original}'? {is_d}")

## N-cubes (Hypercube graphs $Q_n$)

$Q_n$ has $2^n$ vertices (all $n$-bit strings), $n \cdot 2^{n-1}$ edges,
is $n$-regular, bipartite, and has a Hamilton cycle (Gray code) for $n \geq 1$.

In [ ]:
# Stats for Q_1 through Q_5
print(f"{'n':>3}  {'vertices':>9}  {'edges':>8}  {'degree':>7}  Hamilton")
print("-" * 40)
for n in range(1, 6):
    info = hypercube_info(n)
    print(f"{n:>3}  {info['vertices']:>9}  {info['edges']:>8}  {info['degree']:>7}  {info['hamiltonian']}")

# Show vertices and edges of Q_3
n = 3
info = hypercube_info(n)
print(f"\nQ_{n} vertices ({info['vertices']}):")
print(' ', info['vertex_list'])
print(f"Q_{n} edges ({info['edges']}):")
for e in hypercube_edges(n):
    print(f"  {e[0]}, {e[1]}")

# Bipartition of Q_n: vertices with even vs odd number of 1s
verts = hypercube_vertices(n)
even_part = [v for v in verts if v.count('1') % 2 == 0]
odd_part  = [v for v in verts if v.count('1') % 2 == 1]
print(f"\nBipartition of Q_{n}:")
print(f"  Even-parity: {even_part}")
print(f"  Odd-parity:  {odd_part}")

## Inclusion-Exclusion

$|A_1 \cup \cdots \cup A_n| = \sum|A_i| - \sum|A_i \cap A_j| + \cdots$

In [ ]:
# Brute-force inclusion-exclusion over a list of sets
def union_size_by_inclusion_exclusion(sets):
    n = len(sets)
    total = 0
    for r in range(1, n + 1):
        for combo in combinations(range(n), r):
            inter = set.intersection(*(sets[i] for i in combo))
            total += ((-1) ** (r + 1)) * len(inter)
    return total

# Example: how many integers in [1,100] divisible by 2, 3, or 5?
U = set(range(1, 101))
A2 = {x for x in U if x % 2 == 0}
A3 = {x for x in U if x % 3 == 0}
A5 = {x for x in U if x % 5 == 0}

ie_result = union_size_by_inclusion_exclusion([A2, A3, A5])
direct = len(A2 | A3 | A5)
print(f"Divisible by 2, 3, or 5 in [1,100]: {ie_result} (direct: {direct})", "✓" if ie_result == direct else "✗")

## Pigeonhole Principle

If $n$ objects go into $k$ boxes, at least one box has $\lceil n/k \rceil$ objects.
**Template:** find the minimum $n$ to guarantee a pair/group with some property.

In [ ]:
import math

def pigeonhole_min(k, guarantee_per_box):
    """Minimum objects needed so some box has >= guarantee_per_box objects."""
    return k * (guarantee_per_box - 1) + 1

# How many people needed to guarantee 3 share a birthday? (365 days)
k, g = 365, 3
print(f"Min people for 3 sharing a birthday: {pigeonhole_min(k, g)}")

# How many cards drawn from a deck to guarantee 2 of the same suit?
k, g = 4, 2
print(f"Min cards for 2 of same suit: {pigeonhole_min(k, g)}")

# Generalised: guarantee >= g items in one box among k boxes
for (k, g, desc) in [(7, 2, '2 born same weekday'), (12, 2, '2 born same month')]:
    print(f"Min for {desc}: {pigeonhole_min(k, g)}")

## Bit Strings with Constraints

In [ ]:
from itertools import product as iproduct

def bit_strings(n):
    return [''.join(str(b) for b in bits) for bits in iproduct([0,1], repeat=n)]

n = 5
all_bs = bit_strings(n)
print(f"Total {n}-bit strings: {len(all_bs)}")

# Start with 1
start1 = [s for s in all_bs if s[0] == '1']
print(f"Start with 1: {len(start1)}")

# At least two consecutive 1s
consec = [s for s in all_bs if '11' in s]
print(f"Contain '11': {len(consec)}")

# Exactly k ones
k = 3
exact_k = [s for s in all_bs if s.count('1') == k]
print(f"Exactly {k} ones: {len(exact_k)}  (= C({n},{k}) = {comb(n,k)})")

## Multiplicative Inverse & Euler's φ

In [ ]:
# Multiplicative inverse: find x such that a*x ≡ 1 (mod m)
# Exists iff gcd(a, m) = 1
a, m = 7, 26
inv = pow(a, -1, m)   # Python 3.8+ built-in modular inverse
print(f"{a}^(-1) mod {m} = {inv}  (check: {a}*{inv} mod {m} = {a*inv % m})")

# Euler φ(n): count of integers in [1,n] coprime to n
print("\nEuler φ:")
for n in [6, 10, 12, 15, 30]:
    print(f"  φ({n}) = {sp.totient(n)}")

# φ of a prime power: φ(p^k) = p^(k-1)(p-1)
p, k = 7, 3
print(f"\nφ({p}^{k}) = {sp.totient(p**k)}  (formula: {p}^{k-1}·({p}-1) = {p**(k-1)*(p-1)})")

## GCD, LCM, and Extended Euclidean

In [ ]:
a, b = 252, 198
print(f"gcd({a},{b}) = {math.gcd(a,b)}")
print(f"lcm({a},{b}) = {lcm(a,b)}")
print()
# Full EEA table with quotient and Bézout coefficients
extended_euclidean_table(a, b)

## Primes

In [ ]:
print("Primes below 100:", primes_below(100))
print(f"Is 101 prime? {is_prime(101)}")
print(f"Is 102 prime? {is_prime(102)}")

# Prime factorisation
n = 360
print(f"\n{n} = ", end='')
factors = sp.factorint(n)
print(' · '.join(f'{p}^{e}' if e > 1 else str(p) for p, e in sorted(factors.items())))
print(f"φ({n}) = {sp.totient(n)}")

## Set Expression Parser

Write set expressions as strings using: `inter`, `union`, `without` (set difference), `~` or `overline` (complement).
Lets you paste in the MC options directly and get ✓/✗ automatically.

In [ ]:
# Use a concrete universe and named sets
U = set(range(10))
sets = {
    'A': {0, 2, 4, 6, 8},   # evens
    'B': {1, 2, 3, 4, 5},
    'C': {3, 4, 5, 6, 7},
    'U': U,
}

# Operators: inter  union  without (set diff)  ~  overline (complement)
target_expr = "A inter ~(B without C)"
target = evaluate_set_expression(target_expr, sets, U)
print(f"Target = A ∩ ~(B\\C) = {sorted(target)}\n")

# Check a list of MC option expressions, the correct one gets ✓
options = [
    "(A inter ~B) union (A inter C)",   # correct by De Morgan
    "A inter ~B inter C",               # wrong
    "(A union B) inter (A union ~C)",   # wrong
    "A inter ~B union C",               # wrong (precedence)
]
for opt in options:
    result = evaluate_set_expression(opt, sets, U)
    print(f"  {'✓' if result == target else '✗'}  {opt}")
    print(f"       = {sorted(result)}")

# find_equivalent_set_options does the same in one call:
print()
matches = find_equivalent_set_options(target_expr, options, sets, U)
print("Matching options:", matches)

## Tautology Checker

Supported operators: `not`/`¬`/`!`, `and`/`∧`, `or`/`∨`, `=>`/`->`/`→`, `<=>`/`↔`
Variables: any single letter or word.

In [ ]:
exprs = [
    "(p => q) or (not q => not p)",
    "(not p or not q) => (not (p and q))",
    "p <=> q",
    "(p or q or r) and (p or not q or not r)",
    "(p => q) <=> (not q => not p)",  # contrapositive, should be tautology
]

for expr in exprs:
    r = tautology_checker(expr)
    mark = '✓ tautology' if r['is_tautology'] else '✗ not tautology'
    print(f"{mark}  {expr}")

# Full truth table for a specific expression
print()
r = tautology_checker("(p => q) <=> (not p or q)")
print("Truth table for (p→q) ↔ (¬p∨q):")
for row in r['truth_table']:
    vals = ', '.join(f"{k}={int(v)}" for k, v in row['valuation'].items())
    print(f"  {vals}  →  {row['result']}")

## Relation Classifier

Give it a set `S` and a relation `R` (set of pairs), it tells you reflexive/symmetric/etc.
and classifies as partial order, equivalence, Hasse diagram, well order, or none.

In [ ]:
S = {1, 2, 3, 4}

# Divisibility on {1,2,3,4}: a R b iff a divides b
R_div = {(a, b) for a in S for b in S if b % a == 0}
print("Divisibility on {1,2,3,4}:", sorted(R_div))
c = classify_relation(S, R_div)
print("  Reflexive:", c['reflexive'], " Antisymmetric:", c['antisymmetric'],
      " Transitive:", c['transitive'])
print("  =>", c['classification'])

print()
# Congruence mod 2: a R b iff (a-b) divisible by 2
R_mod2 = {(a, b) for a in S for b in S if (a - b) % 2 == 0}
print("≡ mod 2 on {1,2,3,4}:", sorted(R_mod2))
c2 = classify_relation(S, R_mod2)
print("  Symmetric:", c2['symmetric'], " Transitive:", c2['transitive'])
print("  =>", c2['classification'])

## Function Classifier

`classify_function_finite(domain, codomain, func)`, tests well-definedness,
injectivity, surjectivity on a finite sample.

`classify_linear_function_descriptor(str)`, parses `'f: NN -> NN, f(x)=2x+7'` for linear functions.

In [ ]:
# Template: swap in your function and domain/codomain sample
functions = [
    ("f(x) = 2x+7  (NN→NN)",  range(0,20), range(0,50),  lambda x: 2*x+7),
    ("f(x) = x²    (NN→NN)",  range(0,10), range(0,100), lambda x: x**2),
    ("f(x) = x//2  (NN→NN)",  range(0,10), range(0,10),  lambda x: x//2),
    ("f(x) = x mod 3 (NN→{0,1,2})", range(0,20), [0,1,2], lambda x: x % 3),
]

for label, dom, cod, func in functions:
    r = classify_function_finite(dom, cod, func)
    print(f"{label:35s}  {r['classification']}")

# String descriptor for linear functions (parses the text directly)
print()
for desc in [
    "f: NN -> NN, f(x) = 2x + 7",
    "f: ZZ -> ZZ, f(x) = 3x + 1",
    "f: RR -> RR, f(x) = 5x - 2",
]:
    r = classify_linear_function_descriptor(desc)
    print(f"{desc:35s}  {r['classification']}")

## Polynomial GCD & Extended Euclidean

In [ ]:
# gcd only
p1 = Poly(x**3 - 1, x)
p2 = Poly(x**3 + 2*x**2 + 2*x + 1, x)
g = polynomial_gcd(p1, p2)
print("gcd:", g.as_expr())

# Bézout identity, s*p1 + t*p2 = gcd
g, s, t = polynomial_extended_gcd(p1, p2, show_steps=True)

# Full EEA table with quotient column
print()
polynomial_extended_euclidean_table(p1, p2)

## Coefficient of a Monomial + Match MC Options

Expand a polynomial and extract the coefficient of $x^a y^b$,
then check which MC answer option it matches.

In [ ]:
# Option strings must use sympy-parseable syntax.
# Use  binomial(n, k)  for C(n,k), and  **  for exponents.

poly = expand((2*x**3 - y**4)**10)
target_powers = {'x': 15, 'y': 20}

coeff = coefficient_of_monomial(poly, target_powers)
print(f"Coefficient of x^15·y^20 in (2x³-y⁴)^10: {coeff}\n")

# MC options, write them as sympy expressions
options = [
    "-binomial(10,5)*2**5",   # correct
    "binomial(10,5)*2**15",
    "-binomial(10,5)*2**15",
    "0",
    "-binomial(10,5)*2**10",
    "-binomial(10,3)*2**5",
]
matches = coefficient_matches_options(poly, target_powers, options)
print("Matching options:", matches)
print()

# Template: swap in your polynomial and powers
# poly  = expand((x**a - c*y**b)**n)
# coeff = coefficient_of_monomial(poly, {'x': target_x, 'y': target_y})
# coefficient_matches_options(poly, {'x': target_x, 'y': target_y}, your_options)

## System of Congruences (CRT)

Solves $x \equiv a_i \pmod{n_i}$ for pairwise coprime moduli.
Input: list of `[a, n]` pairs.

In [ ]:
# Template, swap in your values
system = [
    [1, 2],   # x ≡ 1 (mod 2)
    [1, 5],   # x ≡ 1 (mod 5)
    [7, 9],   # x ≡ 7 (mod 9)
]
x0, m = congruences_system_solver(system)
print(f"Verify: x0={x0} satisfies each congruence?")
for a, n in system:
    print(f"  {x0} mod {n} = {x0 % n}  (need {a})  {'✓' if x0 % n == a else '✗'}")

## Permutation Constraints, All Options

| Parameter | Meaning |
|---|---|
| `must_contain_all` | all listed substrings must appear |
| `must_contain_any` | at least one listed substring must appear |
| `must_not_contain_any` | none of the listed substrings may appear |
| `exactly_one_of` | exactly one listed substring appears |
| `count_permutations_subsequence` | counts non-contiguous subsequence matches |

In [ ]:
items = 'ABCDE'
total = len(get_permutations(items))
print(f"Total permutations of '{items}': {total}\n")

cases = [
    ("Contain AB (contiguous)",           dict(must_contain_all=['AB'])),
    ("Contain AB or CD",                   dict(must_contain_any=['AB','CD'])),
    ("Contain none of AB, BC, CD",         dict(must_not_contain_any=['AB','BC','CD'])),
    ("Contain exactly one of AB, CD",      dict(exactly_one_of=['AB','CD'])),
    ("Contain both AB and CD",             dict(must_contain_all=['AB','CD'])),
]
for label, kwargs in cases:
    count = count_permutations_with_constraints(items, **kwargs)
    print(f"  {count:4d}  {label}")

print()
for sub in ['AB','ACE','AE']:
    count = count_permutations_subsequence(items, sub)
    print(f"  {count:4d}  '{sub}' as subsequence")

## Division with Remainder

In [ ]:
# Integer division: a = q·b + r
divide_with_remainder(17, 5)
divide_with_remainder(100, 7)

## r-Permutations, r-Combinations & All Subsets

In [ ]:
items = 'ABCD'

# All 2-permutations (order matters)
print("P(4,2):", get_r_permutations(items, 2))

# All 2-combinations (order doesn't matter)
print("C(4,2):", get_r_combinations(items, 2))

# All non-empty subsets
print("All subsets:", get_combinations(items))

## Minimum Selections to Guarantee a Pair (Pigeonhole)

In [ ]:
# How many items must you draw to guarantee a pair summing to target?
# Example: numbers = {1,..,9}, guarantee a pair summing to 10
numbers = list(range(1, 10))
target = 10
n = minimum_selections_for_sum(numbers, target)
print(f"Minimum draws to guarantee a pair summing to {target}: {n}")

## Linear Congruences with Coefficients (ax ≡ c mod n)

In [ ]:
# solve_system_congruences handles ax ≡ c (mod n) — each entry: (coeff, rhs, modulus)
# Example: 2x ≡ 4 (mod 6)  AND  3x ≡ 9 (mod 12)
sol = solve_system_congruences([(2, 4, 6), (3, 9, 12)])
print("Solution:", sol)

# E25 Exam

## Q1, Euler's Totient of $pq$

If $p, q$ prime, $100 < p < q$, how many positive integers less than $pq$ are relatively prime to $pq$?

**Answer:** $pq - p - q + 1 = \varphi(pq)$

**Why:** $\varphi(pq) = (p-1)(q-1) = pq - p - q + 1$.

In [ ]:
# Verify numerically with any pair of primes > 100
p, q = 101, 103 # MUST BE PRIMES > 100
phi = sp.totient(p * q)
formula = p * q - p - q + 1
print(f"phi({p}*{q}) = {phi}")
print(f"pq - p - q + 1 = {formula}")
print("Match:", phi == formula)

# Check which answer option matches symbolically
options = ["p*q - p - q + 1", "p*q - 1", "p*q - p - q - 1"]
print("\nOption check:", check_totient_options(options))

## Q2, Nested Modular Arithmetic

Compute $(4^{100} \bmod 6)^{100} \bmod 10$.

**Answer:** $6$

In [ ]:
# the expression
expr = (4**100 % 6)**100 % 10

print(f"{expr}")

## Q3, Subsets of $\{1, \ldots, 99\}$ (Multi-part)

| Sub-question | Answer |
|---|---|
| How many subsets have 49 elements? | $\binom{99}{49}$ |
| How many subsets have odd # even AND even # odd? | $2^{97}$ |
| How many subsets have odd # odd AND even # even? | $2^{97}$ |
| How many subsets have an odd number of odd numbers? | $2^{98}$ |

In [ ]:
n = 99  # set is {1, 2, ..., 99}
m = 49  # subset size

# Exact size
print(f"Subsets of size 49: C({n}, {m}) = {comb(n, m)}")

# Parity counts
counts = subset_parity_counts_1_to_n(n)
print(f"\nOdd #{'{'}even{'}'}  AND  even #{'{'}odd{'}'}: {counts['odd_even_and_even_odd']}")
print(f"{pretty(factorint(counts['odd_even_and_even_odd'], visual=True))}")

print(f"\nOdd #{'{'}odd{'}'}  AND  even #{'{'}even{'}'}: {counts['odd_odd_and_even_even']}")
print(f"{pretty(factorint(counts['odd_odd_and_even_even'], visual=True))}")

print(f"\nOdd #{'{'}odd{'}'} (any #{'{'}even{'}'}): {counts['odd_odd']}")
print(f"{pretty(factorint(counts['odd_odd'], visual=True))}")

## Q4, System of Congruences (CRT)

$$x \equiv 1 \pmod{2}, \quad x \equiv 1 \pmod{5}, \quad x \equiv 7 \pmod{9}$$

**Answer:** $\{61 + 90k \mid k \in \mathbb{Z}\}$

In [ ]:
# Each pair is [a, n] meaning x ≡ a (mod n).
# Requires pairwise coprime moduli.
congruences_system_solver([
    [1, 2],
    [1, 5],
    [7, 9],
])

## Q5, GCD of Polynomials

Find $\gcd(x^3 - 1,\; x^3 + 2x^2 + 2x + 1)$.

**Answer:** $x^2 + x + 1$

In [ ]:
p1 = Poly(x**3 - 1, x)
p2 = Poly(x**3 + 2*x**2 + 2*x + 1, x)

g = polynomial_gcd(p1, p2)
print("GCD:", g.as_expr())

# Full EEA table
polynomial_extended_euclidean_table(p1, p2)

## Q6, Proof by Induction (Ordering)

**Claim:** $f(n) = \sum_{k=0}^n k \cdot k! = (n+1)! - 1$

**Correct order of fragments: F → B → H → D**

- **F**: Base case $n=0$: $f(0) = 0 = 1! - 1$ ✓
- **B**: Induction step, assume $f(n) = (n+1)!-1$, prove for $n+1$
- **H**: Calculation: $f(n+1) = f(n) + (n+1)(n+1)! = (n+2)!-1$ ✓
- **D**: Conclusion by principle of induction

*(No code needed, pure reasoning.)*

In [ ]:
# Verify the formula f(n) = (n+1)! - 1
def f(n):
    return sum(k * factorial(k) for k in range(n + 1))

formula = lambda n: factorial(n + 1) - 1

# Step 1: Check the base case
base_n = 0
print(f"Base case n={base_n}: f({base_n})={f(base_n)}, formula={formula(base_n)}, OK={f(base_n)==formula(base_n)}")

# Step 2: Check induction step, for each n, assuming formula(n) holds,
# verify that f(n+1) == formula(n+1).
# (This doesn't *prove* it, but catches wrong formulas fast.)
print("\nInduction step spot-checks:")
all_ok = True
for n in range(0, 8):
    lhs = f(n + 1)
    rhs = formula(n + 1)
    ok = lhs == rhs
    all_ok = all_ok and ok
    print(f"  n={n}: f(n+1)={lhs}, formula(n+1)={rhs}, ✓" if ok else f"  n={n}: MISMATCH f(n+1)={lhs} != {rhs}")

print(f"\n{'✓ Formula matches for all tested n, proof structure is plausible.' if all_ok else '✗ Formula is wrong.'}")

# Guide: to test YOUR candidate formula, replace `formula` above and re-run.
# E.g. to test a wrong candidate:
wrong_formula = lambda n: factorial(n + 1)
wrong_ok = all(f(n) == wrong_formula(n) for n in range(8))
print(f"\nWrong formula (n+1)! matches? {wrong_ok}")

## Q7, Piecewise Recursive Sequence

$$a_0=2,\; a_1=3,\; a_n = \begin{cases} a_{n-1}+n & \text{if } n \text{ even} \\ a_{n-1}+2a_{n-2} & \text{if } n \text{ odd} \end{cases}$$

Compute $a_5$.

**Answer:** $37$

In [ ]:
result = recursive_piecewise_value(
    n=5,
    initial_values={0: 2, 1: 3},
    even_rule=lambda n, a: a(n - 1) + n,
    odd_rule=lambda n, a: a(n - 1) + 2 * a(n - 2),
)
print(f"a_5 = {result}")

# Step-by-step trace
def a(n):
    if n == 0: return 2
    if n == 1: return 3
    if n % 2 == 0: return a(n-1) + n
    return a(n-1) + 2*a(n-2)

for i in range(6):
    print(f"  a_{i} = {a(i)}")

## Q8, Bipartite Graph Degree Sequences (Multi-part)

Bipartition $(V_1, V_2)$ with $|V_1|=4$, $|V_2|=5$.

| Degrees $V_1$ | Degrees $V_2$ | Answer |
|---|---|---|
| $1,2,2,2$ | $1,2,2,2,2$ | Does not exist, but will if we increase a degree in $V_1$ |
| $5,5,5,5$ | $4,4,4,4,4$ | Exists without multiple edges |
| $4,4,4,4$ | $5,5,5,5,5$ | Does not exist, but will if we increase a degree in $V_1$ |

In [ ]:
cases = [
    ([1, 2, 2, 2], [1, 2, 2, 2, 2]),   # sum(V1)=7 < sum(V2)=9  → increase V1
    ([5, 5, 5, 5], [4, 4, 4, 4, 4]),   # sums equal, Gale-Ryser OK → simple graph exists
    ([4, 4, 4, 4], [5, 5, 5, 5, 5]),   # sum(V1)=16 < sum(V2)=25 → increase V1
    ([5, 5, 5, 5], [2, 2, 2, 2, 2]),   # sum(V1)=20 > sum(V2)=10 → increase V2
]

for v1, v2 in cases:
    r = bipartite_degree_check(v1, v2)
    print(f"V1={v1}, V2={v2}")
    print(f"  sum(V1)={r['sum_V1']}, sum(V2)={r['sum_V2']}")
    print(f"  => {r['classification']}\n")

## Q9, Vandermonde Identity Lower Bound

$$\binom{m+n}{r} = \sum_{k=a}^{r} \binom{m}{r-k}\binom{n}{k}, \quad 0 < m < r < n$$

**Answer:** $a = r - m$

**Why:** $\binom{m}{r-k}=0$ when $r-k > m$, i.e. $k < r-m$, so summing from $r-m$ avoids zero terms.

In [ ]:
# Vandermonde identity: C(m+n, r) = Σ_{k=a}^{r} C(m, r-k)·C(n, k)  where a = r-m
# In Python, a generator expression reads just like the sigma notation:
#
#   Σ_{k=a}^{r} C(m, r-k)·C(n, k)
#   sum( comb(m, r-k) * comb(n, k)  for k in range(a, r+1) )
#    ↑Σ     ↑ C(m, r-k)    ↑ C(n,k)          ↑ k from a to r

test_cases = [(2, 5, 3), (3, 7, 5), (1, 4, 2)]

for m, n, r in test_cases:
    a = r - m                          # lower bound of summation
    # Left hand side: C(m+n, r)
    lhs = comb(m + n, r)

    # Right hand side: Σ(k=a..r) C(m, r-k)·C(n, k)
    rhs = sum(comb(m, r - k) * comb(n, k) for k in range(a, r + 1))

    status = '✓' if lhs == rhs else '✗'
    print(f"{status} m={m}, n={n}, r={r}: C({m+n},{r}) = Σ(k={a}..{r}) C({m},{r}-k)·C({n},k) = {rhs}")

## Q10, Set Operations

$A=\{0,1,2,4\}$, $B=\{0,1,3,5\}$, $C=\{0,2,3,6\}$, $U=\{0,\ldots,7\}$

Which set equals $((A \cap B) \setminus C) \cup ((B \cap C) \setminus A) \cup ((C \cap A) \setminus B)$?

**Answer:** $\{1, 2, 3\}$

In [ ]:
U = set(range(8)) # 0 indexed, so if 0-7 you do range(8) to include 7
sets = {'A': {0,1,2,4}, 'B': {0,1,3,5}, 'C': {0,2,3,6}, 'U': U}

target_expr = "(A inter B) without C union (B inter C) without A union (C inter A) without B"


result = evaluate_set_expression(target_expr, sets, U)
print("Result:", sorted(result))


## Q11, Predicate Logic Translation

"For every positive rational $x$ there are positive integers $a, b$ with $x = a/b$ and $\gcd(a,b)=1$."

**Answer:** $\forall x \in \mathbb{Q}^+ \; \exists a \in \mathbb{Z}^+ \; \exists b \in \mathbb{Z}^+ \; (x = a/b \wedge G(a,b))$

**Key distinctions:**
- The quantifier on $x$ is $\forall$, not $\exists$.
- $a$ and $b$ are $\exists$, not $\forall$.
- The body is a conjunction ($\wedge$), *not* an implication.
  - $\exists a\,\exists b\,(x = a/b \to G(a,b))$ would be trivially true by picking $a,b$ where $x \neq a/b$.

## Q12, Set Algebra Identity

Which set equals $A \cap \overline{(B \setminus C)}$?

**Answer:** $(A \cap \overline{B}) \cup (A \cap C)$

**Derivation:** $\overline{B \setminus C} = \overline{B \cap \overline{C}} = \overline{B} \cup C$, so $A \cap (\overline{B} \cup C) = (A \cap \overline{B}) \cup (A \cap C)$.

In [ ]:
U = set(range(10))
A = {0, 2, 4, 6, 8}
B = {1, 2, 3, 4, 5}
C = {3, 4, 5, 6, 7}
sets = {'A': A, 'B': B, 'C': C, 'U': U}

target_expr = "A inter ~(B without C)"

# Drop the option strings in from the MC question, the correct one will be marked
options = [
    "(A inter ~B) union (A inter C)",   # correct
    "A inter ~B inter C",
    "(A union B) inter (A union ~C)",
    "A inter ~B union C",
]

target = evaluate_set_expression(target_expr, sets, U)

print(f"Target  A ∩ ~(B\\C) = {sorted(target)}\n")

for opt in options:
    result = evaluate_set_expression(opt, sets, U)
    mark = '✓' if result == target else '✗'
    print(f"  {mark}  {opt}")
    print(f"       = {sorted(result)}")

## Q13, Empty Set Membership

The empty set $\emptyset$ is an element of which sets?

**Answer:** $\{\emptyset\}$

**Trick summary:**
- $\{\emptyset\}$, YES: this set contains exactly one element, which is $\emptyset$.
- $\emptyset$, NO: the empty set has no elements.
- $\{\{\emptyset\}\}$, NO: contains $\{\emptyset\}$, not $\emptyset$.
- $\{x \in \mathbb{R} : x < x\}$ = $\emptyset$, NO: the empty set has no elements.

In [ ]:
truths = empty_set_option_truths()
for name, val in truths.items():
    print(f"  ∅ ∈ {name}: {val}")

## Q14, Tautology Checker

Which is a tautology?

| Expression | Tautology? |
|---|---|
| $(p \to q) \vee (\neg q \to \neg p)$ | No |
| $p \leftrightarrow q$ | No |
| $(\neg p \vee \neg q) \to (\neg(p \wedge q))$ | **Yes** |
| $(p \vee q \vee r) \wedge (p \vee \neg q \vee \neg r)$ | No |

In [ ]:
# why is this a list of tuples?
expressions = [
    ("(p => q) or (not q => not p)"),
    ("p <=> q"),
    ("(not p or not q) => (not (p and q))"),
    ("(p or q or r) and (p or not q or not r)"),
    ("p <=> p")
]

for expr in expressions:
    r = tautology_checker(expr)
    print(f"{expr:35s}  tautology: {r['is_tautology']}")

## Q15, Circular Seating at Two Tables

$3n$ people, one table of $n$ seats, one of $2n$ seats.

| Equivalence | Answer |
|---|---|
| Same left **and** right neighbor | $\dfrac{(3n)!}{2n^2}$ |
| Same neighbors (no L/R distinction) | $\dfrac{(3n)!}{8n^2}$ |

In [ ]:
# 3n people; table 1 has n seats, table 2 has 2n seats
n = 3
people = 3 * n
table1 = n
table2 = 2 * n

print(f"Setup: {people} people, tables of {table1} and {table2} seats\n")

lr   = seating_count_two_tables(n, distinguish_left_right=True)
no_lr = seating_count_two_tables(n, distinguish_left_right=False)

print(f"Same when same L+R neighbor : {lr}")
print(f"Same when same neighbors    : {no_lr}")

# Check against the MC answer option formulas
import sympy as sp
options = {
    "(3n)!/(2n²)": sp.Rational(math.factorial(3*n), 2*n**2),
    "(3n)!/(8n²)": sp.Rational(math.factorial(3*n), 8*n**2),
    "(3n)!/(4n²)": sp.Rational(math.factorial(3*n), 4*n**2),
    "(3n)!/(2n) ": sp.Rational(math.factorial(3*n), 2*n),
}
print(f"\nOption check for n={n}:")
for label, val in options.items():
    lr_mark   = '✓' if val == lr    else ' '
    nolr_mark = '✓' if val == no_lr else ' '
    print(f"  {label} = {val}   L/R:{lr_mark}  no-L/R:{nolr_mark}")

## Q16, Function Classification (Multi-part)

| Function | Classification |
|---|---|
| $f:\mathbb{R}\to\mathbb{Z},\; f(x)=2\lfloor x/2 \rfloor$ | Well-defined but neither |
| $f:\mathbb{R}\to\{x\geq 0\},\; f(x)=\sqrt{x^2}$ | Surjective |
| $f:\mathbb{N}\to\mathbb{N},\; f(x)=2x+7$ | Injective |
| $f:\mathbb{N}\to\mathbb{N},\; f(x)=x-x^2$ | Not well-defined (negative for $x>1$) |
| $f:\mathbb{R}\to\mathbb{R},\; f(x)=x$ if $x\in\mathbb{Q}$, $-x$ if $x\notin\mathbb{Q}$ | Both (bijective) |

In [ ]:
import math, fractions

# TODO: Stop printning large seperators, and remove hardcoded values, the classify function should be able to check the surjectivity, injectivity and well-definedness of the function.

print("1. f: ℝ → ℤ,  f(x) = 2⌊x/2⌋")
# Range: only even integers → not surjective onto ℤ
# Not injective: f(0) = f(0.5) = 0
samples = [-3, -2, -1, 0, 0.5, 1, 1.5, 2, 3]
vals = [2 * math.floor(x / 2) for x in samples]
print(f"  sample outputs: {vals}")
print("  Not injective (0 and 0.5 both → 0), not surjective (1 unreachable)")
print("  => Well-defined but NEITHER")

print("─" * 60)
print("2. f: ℝ → {x≥0},  f(x) = √(x²) = |x|")
print("  Surjective: every y≥0 is hit (f(y)=y)")
print("  Not injective: f(1) = f(-1) = 1")
print("  => SURJECTIVE but not injective")

print("─" * 60)
print("3. f: ℕ → ℕ,  f(x) = 2x + 7")
r = classify_function_finite(range(0, 30), range(0, 100), lambda x: 2*x + 7)
print(f"  {r['classification']} (range is only odd integers ≥ 7)")

print("─" * 60)
print("4. f: ℕ → ℕ,  f(x) = x − x²")
r2 = classify_function_finite(range(0, 10), range(0, 100), lambda x: x - x**2)
print(f"  {r2['classification']}, counterexample: x={r2['counterexample'][0]} → {r2['counterexample'][1]}")

print("─" * 60)
print("5. f: ℝ → ℝ,  f(x) = x if x∈ℚ, else −x")
# Injective: if both rational f(x)=x≠y=f(y); if x∈ℚ,y∉ℚ then x≠-y (different rationality)
# Surjective: any z∈ℚ achieved by z; any z∉ℚ achieved by -z (also irrational)
print("  Injective + surjective => BOTH (bijective)")
# Spot-check on rationals:
dom = [fractions.Fraction(i) for i in range(-5, 6)]
r5 = classify_function_finite(dom, list(range(-5, 6)), lambda x: x)
print(f"  Rational domain sample: {r5['classification']}")

In [ ]:
# Linear descriptor parser for simple cases
descriptor = "f: NN -> NN, f(x) = 2x + 7"
r = classify_linear_function_descriptor(descriptor)
print(descriptor)
print(" ->", r['classification'])
print()

## Q17, Permutations of ABCDE with Constraints (Multi-part)

| Constraint | Answer |
|---|---|
| Contain none of AB, BC, CD | $64$ |
| Contain ACE as subsequence | None of these ($= 20$) |
| Contain precisely one of AB, CD | $36$ |

In [ ]:
perms = get_permutations('ABCDE')
print(f"Total permutations of ABCDE: {len(perms)}\n")

# Contiguous substring constraints
none_abc = count_permutations_with_constraints('ABCDE', must_not_contain_any=['AB', 'BC', 'CD'])
print(f"None of AB, BC, CD (contiguous):  {none_abc}")

one_of_ab_cd = count_permutations_with_constraints('ABCDE', exactly_one_of=['AB', 'CD'])
print(f"Precisely one of AB, CD:           {one_of_ab_cd}")

contains_abc = count_permutations_with_constraints('ABCDE', must_contain_all=['ABC'])
print(f"Contain ABC (contiguous):          {contains_abc}")

ab_or_cd = count_permutations_with_constraints('ABCDE', must_contain_any=['AB', 'CD'])
print(f"Contain AB or CD:                  {ab_or_cd}")

# Non-contiguous subsequence constraint
# Use count_permutations_subsequence: A appears before C, C before E (anywhere)
ace_count = count_permutations_subsequence('ABCDE', 'ACE')
print(f"\nACE as subsequence (A<C<E pos):    {ace_count}  <- 'None of these' (not in options 12,24,36,48,50,64)")

ab_subseq = count_permutations_subsequence('ABCDE', 'AB')
print(f"AB as subsequence (A before B):    {ab_subseq}")

# ── Template: swap in your own letters/patterns ──────────────────────────
# items      = 'ABCDE'                          # change the set
# patterns   = ['XY', 'YZ']                     # contiguous substrings to check
# subseq     = 'ACE'                            # non-contiguous subsequence
# count_permutations_with_constraints(items, must_not_contain_any=patterns)
# count_permutations_subsequence(items, subseq)

## Q18, Relation Classification on $\{a,b,c,d\}$ (Multi-part)

| Relation | Classification |
|---|---|
| $\{(a,a),(a,b),(a,c),(a,d)\}$ | None |
| $\{(a,b),(b,c),(c,d)\}$ | Hasse / covering relation |
| $\{(a,a),(b,b),(c,c),(d,d),(a,d),(d,a)\}$ | Equivalence relation |
| $\{(a,a),(b,b),(c,c),(d,d),(d,c)\}$ | Partial order |
| $\{(a,a),(b,b),(c,c),(d,d),(a,b),(a,c),(a,d),(b,c),(b,d),(c,d)\}$ | Well order |

In [ ]:
S = {'a', 'b', 'c', 'd'}

relations = [
    {('a','a'),('a','b'),('a','c'),('a','d')},
    {('a','b'),('b','c'),('c','d')},
    {('a','a'),('b','b'),('c','c'),('d','d'),('a','d'),('d','a')},
    {('a','a'),('b','b'),('c','c'),('d','d'),('d','c')},
    {('a','a'),('b','b'),('c','c'),('d','d'),
     ('a','b'),('a','c'),('a','d'),('b','c'),('b','d'),('c','d')},
]

for i, R in enumerate(relations, 1):
    c = classify_relation(S, R)
    props = [k for k in ['reflexive','irreflexive','symmetric','antisymmetric','transitive'] if c[k]]
    print(f"R{i}: {c['classification']}")
    print(f"     props: {props}\n")

## Q19, Recursive Tiling of $n \times 2$ Board

Number of ways to tile an $n \times 2$ board with $2 \times 1$ tiles.

**Answer:** $f(0)=1,\; f(1)=1,\; f(n)=f(n-1)+f(n-2)$ for $n \geq 2$

This is the Fibonacci sequence.

In [ ]:
# Brute-force: tile n×2 board with 2×1 tiles
# Fact: exactly the Fibonacci numbers (f(n)=f(n-1)+f(n-2), f(0)=f(1)=1)
def brute_force_tiling(n, memo={}):
    if n in memo: return memo[n]
    if n <= 1: return 1
    memo[n] = brute_force_tiling(n-1) + brute_force_tiling(n-2)
    return memo[n]

brute = [brute_force_tiling(n) for n in range(9)]
print(f"Brute-force f(0..8): {brute}\n")

# Test each candidate recurrence from the MC options
def test_recurrence(name, base_cases, rule, n_max=8):
    memo = dict(base_cases)
    def f(n):
        if n not in memo: memo[n] = rule(n, f)
        return memo[n]
    candidate = [f(n) for n in range(n_max + 1)]
    ok = candidate == brute[:n_max + 1]
    print(f"  {'✓ CORRECT' if ok else '✗ wrong  '} {name}")
    print(f"            {candidate}")

candidates = [
    ("f(0)=1, f(1)=1, f(n)=f(n-1)+f(n-2)", {0:1, 1:1}, lambda n,f: f(n-1)+f(n-2)),
    ("f(0)=0, f(1)=1, f(n)=f(n-1)+f(n-2)", {0:0, 1:1}, lambda n,f: f(n-1)+f(n-2)),
    ("f(0)=1, f(1)=1, f(n)=2·f(n-2)",       {0:1, 1:1}, lambda n,f: 2*f(n-2)),
    ("f(0)=0, f(1)=1, f(n)=2·f(n-2)",       {0:0, 1:1}, lambda n,f: 2*f(n-2)),
    ("f(0)=1, f(1)=1, f(n)=f(n-1)·f(n-2)", {0:1, 1:1}, lambda n,f: f(n-1)*f(n-2)),
]
for name, base, rule in candidates:
    test_recurrence(name, base, rule)

## Q20, Coefficient of $x^{15}y^{20}$ (Multi-part)

| Polynomial | Coefficient of $x^{15}y^{20}$ |
|---|---|
| $(2x^3 - y^4)^{10}$ | $-\binom{10}{5} \cdot 2^5$ |
| $(1 - 2x^3 y^4)^{10}$ | $-\binom{10}{5} \cdot 2^5$ |
| $(x^3 - 2y^4)^{10}$ | $-\binom{10}{5} \cdot 2^5$ |

In [ ]:
from math import comb as _comb

# TODO: use LaTeX formatting so output is readable.
def format_coeff(coeff):
    """Try to express an integer as ±C(n,k)·b^e for small n,k,b,e."""
    c = int(coeff)
    sign, c_abs = (-1, -c) if c < 0 else (1, c)
    best = str(coeff)  # fallback
    for b in range(2, 12):
        for e in range(0, 25):
            bpow = b ** e
            if bpow > c_abs: break
            if c_abs % bpow == 0:
                rem = c_abs // bpow
                for nn in range(0, 20):
                    for kk in range(0, nn + 1):
                        if _comb(nn, kk) == rem:
                            s = '-' if sign < 0 else ''
                            best = f"{s}C({nn},{kk})·{b}^{e}" if e > 0 else f"{s}C({nn},{kk})"
                            return best  # first (smallest) match
    return best

target_powers = {'x': 15, 'y': 20}

polys = {
    "(2x³ - y⁴)¹⁰": expand((2*x**3 - y**4)**10),
    "(1 - 2x³y⁴)¹⁰": expand((1 - 2*x**3*y**4)**10),
    "(x³ - 2y⁴)¹⁰":  expand((x**3 - 2*y**4)**10),
}

print(f"Looking for coefficient of x^15 · y^20\n")
for label, poly in polys.items():
    coeff = coefficient_of_monomial(poly, target_powers)
    print(f"  {label}: {coeff}  =  {format_coeff(coeff)}")

## Q21, Logical Equivalences for "$a$ and $b$ are relatively prime"

**Answer:** All three are equivalent.

The three statements (over positive integers):
1. $\neg(\exists c\;(c \mid a \wedge c \mid b \wedge c > 1))$
2. $\forall c\;(\neg(c \mid a) \vee \neg(c \mid b) \vee (c \leq 1))$
3. $\forall c\;((c \mid a \wedge c \mid b) \implies (c \leq 1))$

Statements 1 and 2 are De Morgan duals. Statement 3 is the implication form of 2.
All three say: no common divisor exceeds 1.

In [ ]:
def check_coprimality_equiv(stmt_func, name, test_range=30):
    """Verify stmt_func(a,b) agrees with gcd(a,b)==1 for all pairs in [1, test_range]."""
    for a in range(1, test_range + 1):
        for b in range(1, test_range + 1):
            expected = math.gcd(a, b) == 1
            if stmt_func(a, b) != expected:
                print(f"  ✗ {name}  FAILS at ({a},{b}): got {stmt_func(a,b)}, expected {expected}")
                return
    print(f"  ✓ {name}")

# The three statements from Q21
stmt1 = lambda a,b: not any(a%c==0 and b%c==0 and c>1 for c in range(2, min(a,b)+1))
stmt2 = lambda a,b: all(a%c!=0 or b%c!=0 or c<=1 for c in range(1, min(a,b)+1))
stmt3 = lambda a,b: all(not(a%c==0 and b%c==0) or c<=1 for c in range(1, min(a,b)+1))


print("Checking against gcd(a,b)==1 for all pairs in [1,30]²:\n")
check_coprimality_equiv(stmt1, "~∃c(c|a ∧ c|b ∧ c>1)")
check_coprimality_equiv(stmt2, "∀c(~(c|a) ∨ ~(c|b) ∨ c≤1)")
check_coprimality_equiv(stmt3, "∀c((c|a ∧ c|b) → c≤1)")

# Template: add your own statement here
# my_stmt = lambda a, b: ...  <your predicate logic here>
# check_coprimality_equiv(my_stmt, 'description')